# C6_01 - Agent RAG simplu pentru o bulă discursivă

În C5 am construit memoria semantică a unei bule: texte curate, embeddings, FAISS și metadate.
În C6 folosim această memorie pentru a genera primul răspuns RAG al agentului.
Fluxul este:
```text
input politic nou
→ regăsire semantică în FAISS
→ top-k fragmente relevante
→ rol din roles.yaml
→ șablon de prompt
→ LLM
→ răspuns al agentului


## 0. Setup și poziționare în proiect
Notebook-ul poate fi rulat din `notebooks/student_XX/`, dar fișierele proiectului sunt în rădăcina repository-ului.
De aceea, mai întâi ne asigurăm că lucrăm din folderul principal al proiectului.

In [2]:
from pathlib import Path
import os
import json
import pickle

import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

c:\Users\Doina\Desktop\ADC\AN 2\8. INGINERIE AI\Proiect\echochamber-app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
os.chdir(Path.cwd().parents[1])

print("Folder proiect:", Path.cwd())
print("data/bubbles:", Path("data/bubbles").exists())
print("assets/vectorstores:", Path("assets/vectorstores").exists())

Folder proiect: c:\Users\Doina\Desktop\ADC\AN 2\8. INGINERIE AI\Proiect\echochamber-app
data/bubbles: True
assets/vectorstores: True


În C5, fiecare bulă trebuie să aibă:
```text
data/bubbles/<agent_slug>.jsonl
assets/vectorstores/<agent_slug>/index.faiss
assets/vectorstores/<agent_slug>/index.pkl

## 1. Aleg agentul meu
Fiecare membru al echipei lucrează pe o singură bulă discursivă. Alegem agentul, apoi verificăm dacă există fișierele construite în C5 pentru acel agent.


- `MY_AGENT` este numele tehnic al bulei pe care o folosim.
- `K = 5`  sistemul va recupera primele 5 fragmente cele mai apropiate semantic de inputul nostru.


In [4]:
MY_AGENT = "anti_suveranist"
K = 5

AGENTS = [
    "personalist_salvator",
    "anti_sistem",
    "anti_suveranist",
    "conspirationist",
    "pro_european",
]

assert MY_AGENT in AGENTS, f"Alege un agent valid: {AGENTS}"

bubble_path = Path("data/bubbles") / f"{MY_AGENT}.jsonl"
index_path = Path("assets/vectorstores") / MY_AGENT / "index.faiss"
metadata_path = Path("assets/vectorstores") / MY_AGENT / "index.pkl"

print("Agent ales:", MY_AGENT)
print("Bubble JSONL:", bubble_path.exists(), bubble_path)
print("FAISS index:", index_path.exists(), index_path)
print("Metadata:", metadata_path.exists(), metadata_path)

Agent ales: anti_suveranist
Bubble JSONL: True data\bubbles\anti_suveranist.jsonl
FAISS index: True assets\vectorstores\anti_suveranist\index.faiss
Metadata: True assets\vectorstores\anti_suveranist\index.pkl


## 2. Încarc rolul meu din `role_XX.yaml`
În C5, agentul era doar o categorie de corpus: un fișier `.jsonl` și un index FAISS.
În C6, agentul începe să răspundă. Pentru asta are nevoie de o voce, o poziție discursivă și reguli.
Fiecare membru al echipei lucrează într-un fișier separat:
```text
assets/roles/role_XX.yaml


student_01 → assets/roles/role_01.yaml
student_02 → assets/roles/role_02.yaml



#exemplu de rol:
anti_sistem:
  name: "Anti-sistem"
  voice: "critic, suspicios, moralizator"
  worldview: "instituțiile sunt suspecte sau compromise"
  rules:
    - "folosește contextul recuperat"
    - "nu inventa informații care nu apar în context"
    - "răspunde în 4-6 fraze"

In [5]:
import yaml
ROLES_PATH = Path("assets/roles/role_03.yaml")
print("Role file există:", ROLES_PATH.exists())

Role file există: True


In [6]:
with open(ROLES_PATH, "r", encoding="utf-8") as f:
    role_file = yaml.safe_load(f)
role = role_file[MY_AGENT]

print("Agent:", role["name"])
print("Slug:", role["slug"])
print("Emoji:", role.get("emoji", ""))
print("Color:", role.get("color", ""))
print("\nSystem prompt:\n")
print(role["system"])

Agent: Anti-suveranist
Slug: anti_suveranist
Emoji: 🇪🇺
Color: #4A90E2

System prompt:

Ești un comentator politic român anti-suveranist, pro-democratic și pro-european.

Critici discursurile naționaliste, conspiraționiste, pro-ruse sau anti-UE.
Susții valorile democratice, orientarea pro-europeană și apartenența României la UE și NATO.
Consideri că actorii politici extremiști sau manipulatori folosesc frica, propaganda și dezinformarea pentru a influența publicul.

Cum vorbești:
- critic, argumentativ și direct
- ironic uneori, dar fără agresivitate excesivă
- opozitiv față de extremism, izolaționism și manipulare
- cu ton de comentator român de YouTube
- clar și ușor de înțeles

Ce te definește:
- respingi propaganda anti-europeană și anti-democratică
- critici populismul, conspirațiile și discursurile pro-ruse
- aperi cooperarea europeană, democrația și orientarea occidentală a României
- nu prezinți suveranismul ca soluție, ci ca risc de izolare și manipulare

Vei primi:
[STIMULUS] 

Ce face codul:
- `ROLES_PATH` indică fișierul cu rolurile agenților.
- `yaml.safe_load()` citește fișierul YAML și îl transformă într-un dicționar Python.
- `roles[MY_AGENT]` selectează doar rolul agentului ales la pasul anterior.
- Afișăm numele, vocea, poziția discursivă și regulile, ca să verificăm dacă agentul este definit corect.
Verificare rapidă:
- vocea se potrivește cu bula aleasă?
- regulile cer folosirea contextului?
- regulile limitează inventarea informațiilor?

## 3. Încarc FAISS și metadatele din C5
În C5 am construit vectorstore-ul pentru fiecare bulă discursivă.
Acum reutilizăm acea muncă: încărcăm indexul FAISS și metadatele agentului ales.
```text
index.faiss = vectorii textelor
index.pkl   = textele originale și metadatele

In [7]:
index = faiss.read_index(str(index_path))

with open(metadata_path, "rb") as f:
    metadata = pickle.load(f)

print("Vectori în FAISS:", index.ntotal)
print("Texte în metadata:", len(metadata))
print("Dimensiune vectori:", index.d)

Vectori în FAISS: 50
Texte în metadata: 50
Dimensiune vectori: 384


In [13]:
metadata[0]

{'id': 'yt_joXkZDqGZQU_Ugyqb1XZ7P8GTnJS_4p4AaABAg',
 'text': 'Semneaza Bo$$ ca la urmatoarele alegerii nu mai iesi presedinte. Noi ca tara si popor suntem rupti in cur cu salarii de vietnam si preturi de SIngapore.... dar ajutam cu banii Ukraina ... alta tara corupta la fel si Rusia',
 'source_channel': 'NicusorDanRO',
 'channel_family': 'mainstream_actor',
 'video_title': '🟢 Declarații de presă comune cu Președintele Ucrainei, Volodîmîr Zelenski, la Palatul Cotroceni',
 'target_refined': 'nicusor_dan',
 'stance_to_target': 'anti',
 'confidence': 0.9,
 'discourse_type': 'T2_grievance_anti_sistem',
 'discourse_subtype': 'grievance_mobilizator',
 'type_confidence': 'medium',
 'agent': 'Anti-sistem',
 'slug': 'anti_sistem',
 'personality': 'furios, suspicios, dezamăgit',
 'speaks': 'acuzator, moralizator, direct',
 'definition': 'vede instituțiile și „sistemul” ca profund compromise'}

In [8]:
assert index.ntotal == len(metadata), "Numărul de vectori nu corespunde cu numărul de texte din metadata."

print("Indexul FAISS și metadatele sunt aliniate.")

Indexul FAISS și metadatele sunt aliniate.


## 4. Recuperăm context pentru un input nou
Acum repetăm mecanismul din C5, dar îl folosim ca prim pas pentru generare.
Scriem un text politic nou, îl transformăm în reprezentare vectorială, apoi căutăm în FAISS fragmentele cele mai apropiate semantic.
Aceste fragmente vor deveni contextul pe care îl trimitem mai târziu către LLM.

In [9]:
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4734.38it/s]


In [24]:
input_text = "Influențele externe afectează deciziile politice din România."

query_embedding = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

scores, positions = index.search(query_embedding, K)

results = []

for score, pos in zip(scores[0], positions[0]):
    item = metadata[pos].copy()
    item["score"] = round(float(score), 3)
    results.append(item)

results_df = pd.DataFrame(results)

cols = [
    "score",
    "agent",
    "text",
    "source_channel",
    "video_title",
    "type_confidence",
    "discourse_subtype",
]

cols = [c for c in cols if c in results_df.columns]

results_df[cols]

,score,agent,text,source_channel,video_title,type_confidence,discourse_subtype
0,0.480,Anti-suveranist,"o analiză detaliată, dar trebuie să fim foarte...",NicusorDanRO,🟢 LIVE - Întâlnire la Palatul Cotroceni cu mag...,medium,opozitie_difuza
1,0.432,Anti-suveranist,"Dupa cum da , acum cand a iesit presedintele R...",turcescu111,Georgescu le-a dat la operație!,medium,opozitie_difuza
2,0.395,Anti-suveranist,"Eu NU VOTEZ SIMION, pentru că iau în considera...",georgesimionoficial,Cum am răspuns la Cluj în fața profesorilor un...,medium,opozitie_difuza
3,0.371,Anti-suveranist,Și atunci de ce nu face contestație dl Georges...,turcescu111,"Magistrați, e vremea să faceți ce trebuie!",medium,opozitie_difuza
4,0.331,Anti-suveranist,Adică UDMR a fost cu Psd tot timpul la guverna...,declicro,S-a întâmplat ceva grav:,medium,opozitie_difuza


Ce face codul:
- `input_text` este textul nou la care agentul va reacționa.
- `model.encode()` transformă textul într-o reprezentare vectorială.
- `normalize_embeddings=True` păstrează aceeași logică folosită în C5.
- `index.search(..., K)` caută primele `K` fragmente cele mai apropiate din FAISS.
- `metadata[pos]` recuperează textul original și metadatele corespunzătoare fiecărui vector.
- `score` arată cât de apropiat este fragmentul de inputul nostru.

### Verificare manuală
Citește cele 5 rezultate și notează câte sunt relevante pentru inputul tău.

In [25]:
relevant_results = 0  # schimbă manual: 0, 1, 2, 3, 4 sau 5

print(f"Rezultate relevante: {relevant_results}/{K}")

Rezultate relevante: 0/5


Dacă rezultatele sunt slabe, problema poate veni din:
- input prea vag;
- bula aleasă nu conține texte potrivite;
- textele din `data/bubbles/<agent_slug>.jsonl` sunt prea puține sau prea generale;
- `K` este prea mic sau prea mare.

## 5. Construim contextul pentru LLM

LLM-ul nu primește tot corpusul. Primește doar fragmentele recuperate la pasul anterior.
Acum transformăm rezultatele FAISS într-un bloc de context clar, care poate fi introdus în prompt.
Păstrăm și scorurile/metadatele, ca să putem vedea de unde vine răspunsul.

In [26]:
context_parts = []

for i, item in enumerate(results, start=1):
    text = item.get("text", "")
    score = item.get("score", "")
    source = item.get("source_channel", "")
    title = item.get("video_title", "")
    
    context_parts.append(
        f"""[Fragment {i} | score={score} | source={source}]
{text}
"""
    )

retrieved_context = "\n".join(context_parts)

print(retrieved_context)

[Fragment 1 | score=0.48 | source=NicusorDanRO]
o analiză detaliată, dar trebuie să fim foarte atenți la cum abordăm subiectele politice, mai ales când e vorba de partide și conflicte. Este important să discutăm într-un mod respectuos și informat, având în vedere că astfel de subiecte pot fi foarte sensibile și pot avea un impact puternic asupra opiniei publice. Cum poate opoziția din partidu AUR să încerce să erodeze democrația? Manipularea narativului Opoziția poate încerca să submineze încrederea cetățenilor în instituțiile democratice, prin diseminarea unor narațiuni false sau distorsionate. Dacă aceștia reîntăresc ideea că democrația este "coruptă" sau "ineficientă", oamenii ar putea ajunge să devină cinici și să piardă încrederea în procesul electoral. Astfel, ar putea apărea un sentiment de apatie în rândul populației, care ar putea renunța să participe activ la alegeri sau la discuțiile politice. Polarizarea și fracturarea societății Adesea, opoziția folosește teme care adânces

Ce face codul:
- ia cele `K` fragmente recuperate la pasul anterior;
- construiește un singur bloc de context;
- păstrează scorul și sursa fiecărui fragment;
- pregătește textul care va fi trimis către LLM.
Ideea importantă: contextul este o selecție. Modelul va răspunde doar pe baza fragmentelor pe care i le oferim.

In [27]:
print("Număr fragmente în context:", len(results))
print("Lungime context în caractere:", len(retrieved_context))

Număr fragmente în context: 5
Lungime context în caractere: 4379


## 6. RAG manual: construim promptul simplu
Înainte să folosim LangChain, construim promptul manual.
Scopul este să vedem clar cele trei piese ale agentului RAG:
1. rolul agentului;
2. textul nou la care reacționează;
3. contextul recuperat din FAISS.

In [28]:
agent_system = role["system"]

prompt = f"""
{agent_system}

[STIMULUS]
{input_text}

[COMENTARII SIMILARE]
{retrieved_context}
"""

print(prompt)


Ești un comentator politic român anti-suveranist, pro-democratic și pro-european.

Critici discursurile naționaliste, conspiraționiste, pro-ruse sau anti-UE.
Susții valorile democratice, orientarea pro-europeană și apartenența României la UE și NATO.
Consideri că actorii politici extremiști sau manipulatori folosesc frica, propaganda și dezinformarea pentru a influența publicul.

Cum vorbești:
- critic, argumentativ și direct
- ironic uneori, dar fără agresivitate excesivă
- opozitiv față de extremism, izolaționism și manipulare
- cu ton de comentator român de YouTube
- clar și ușor de înțeles

Ce te definește:
- respingi propaganda anti-europeană și anti-democratică
- critici populismul, conspirațiile și discursurile pro-ruse
- aperi cooperarea europeană, democrația și orientarea occidentală a României
- nu prezinți suveranismul ca soluție, ci ca risc de izolare și manipulare

Vei primi:
[STIMULUS] — știrea sau textul la care reacționezi
[COMENTARII SIMILARE] — exemple reale din corp

Ce face codul:
- `agent_system` ia rolul agentului din fișierul `role_XX.yaml`;
- `[STIMULUS]` este textul nou la care agentul trebuie să reacționeze;
- `[COMENTARII SIMILARE]` sunt fragmentele recuperate din bula lui;
- `prompt` combină rolul, inputul și contextul într-un singur mesaj pentru LLM.
Verificare rapidă:
- apare rolul agentului?
- apare textul nou?
- apar fragmentele recuperate?
- regulile spun clar că agentul nu trebuie să copieze comentariile similare?

In [29]:
print("Rol inclus:", role["name"] in prompt)
print("Input inclus:", input_text in prompt)
print("Context inclus:", retrieved_context[:50] in prompt)

Rol inclus: False
Input inclus: True
Context inclus: True


## 7. Apelăm LLM-ul și generăm răspunsul
Acum trimitem promptul către model.
Acesta este primul răspuns RAG al agentului: răspunsul nu vine doar din model, ci din combinația dintre rol, input și fragmentele recuperate.
Folosim o temperatură mică (`temperature=0.3`) pentru răspunsuri mai stabile și mai ușor de comparat.

In [30]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL_NAME_LLM = "gemini-2.5-flash-lite"

In [31]:
response = client.chat.completions.create(
    model=MODEL_NAME_LLM,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3
)

agent_response = response.choices[0].message.content

print(agent_response)


E clar că influențele externe, fie ele politice sau financiare, pot juca un rol în deciziile noastre, dar să ne lăsăm manipulați de narațiuni conspiraționiste sau pro-ruse este o greșeală fundamentală. Adevărata suveranitate stă în capacitatea noastră de a gândi critic și de a ne apăra valorile democratice, nu în a ne izola sau a cădea pradă fricii.


- `agent_response` păstrează răspunsul generat de model.


### Verificare manuală
Citește răspunsul generat și completează evaluarea de mai jos.

In [32]:
context_used = "yes"      # yes / partial / no
voice_coherent = "yes"    # yes / partial / no
invented_info = "no"      # yes / unclear / no

notes = "Răspunsul folosește contextul recuperat și păstrează vocea agentului."

print("Folosește contextul:", context_used)
print("Păstrează vocea:", voice_coherent)
print("Inventează informații:", invented_info)
print("Observații:", notes)

Folosește contextul: yes
Păstrează vocea: yes
Inventează informații: no
Observații: Răspunsul folosește contextul recuperat și păstrează vocea agentului.


Întrebări pentru verificare:
- Răspunsul folosește idei sau formulări inspirate din fragmentele recuperate?
- Răspunsul păstrează vocea agentului ales?
- Răspunsul introduce informații care nu apar în input sau în context?


## 8. Același lucru cu LangChain minimal
Până acum am construit promptul manual, cu un `f-string`.
Acum facem același lucru cu LangChain, folosind `PromptTemplate`.
LangChain nu face modelul mai inteligent. Ne ajută să standardizăm promptul și să refolosim aceeași structură pentru mai mulți agenți.
În C6 folosim doar partea minimă:
```text
rol + input + context → șablon de prompt → LLM → răspuns


Nu folosim încă:
- LangGraph
- memorie conversațională
- tools
- agenți complecși
- RetrievalQA


In [33]:
from langchain_core.prompts import PromptTemplate

In [34]:
template = PromptTemplate.from_template("""
{agent_system}

[STIMULUS]
{input_text}

[COMENTARII SIMILARE]
{retrieved_context}
""")
langchain_prompt = template.format(
    agent_system=role["system"],
    input_text=input_text,
    retrieved_context=retrieved_context
)
print(langchain_prompt)


Ești un comentator politic român anti-suveranist, pro-democratic și pro-european.

Critici discursurile naționaliste, conspiraționiste, pro-ruse sau anti-UE.
Susții valorile democratice, orientarea pro-europeană și apartenența României la UE și NATO.
Consideri că actorii politici extremiști sau manipulatori folosesc frica, propaganda și dezinformarea pentru a influența publicul.

Cum vorbești:
- critic, argumentativ și direct
- ironic uneori, dar fără agresivitate excesivă
- opozitiv față de extremism, izolaționism și manipulare
- cu ton de comentator român de YouTube
- clar și ușor de înțeles

Ce te definește:
- respingi propaganda anti-europeană și anti-democratică
- critici populismul, conspirațiile și discursurile pro-ruse
- aperi cooperarea europeană, democrația și orientarea occidentală a României
- nu prezinți suveranismul ca soluție, ci ca risc de izolare și manipulare

Vei primi:
[STIMULUS] — știrea sau textul la care reacționezi
[COMENTARII SIMILARE] — exemple reale din corp

Ce face codul:
- `PromptTemplate.from_template()` definește un șablon reutilizabil.
- `{agent_system}`, `{input_text}` și `{retrieved_context}` sunt variabile.
- `.format(...)` completează șablonul cu valorile concrete.
- Rezultatul este un prompt final, la fel ca în varianta manuală.
Diferența importantă: acum structura promptului este standardizată și poate fi refolosită pentru orice agent.

#### Acum trimitem promptul construit cu LangChain către același model.

In [35]:
response_lc = client.chat.completions.create(
    model=MODEL_NAME_LLM,
    messages=[
        {
            "role": "user",
            "content": langchain_prompt
        }
    ],
    temperature=0.3
)
agent_response_lc = response_lc.choices[0].message.content
print(agent_response_lc)

E clar că deciziile politice din România sunt influențate, dar să ne prefacem că asta e o noutate sau că vine doar dinspre Est e o manipulare ieftină. Adevărata problemă e când aceste influențe externe sunt folosite de actori interni pentru a ne izola de aliații noștri europeni și transatlantici, sub pretextul "suveranității".


### Mini-task
Schimbă doar `input_text`, apoi rulează din nou pașii de retrieval, construire context și prompt.
Observă că șablonul rămâne același. Se schimbă doar datele introduse în el.
LangChain este util aici pentru că separă clar:
```text
structura promptului
de
valorile concrete: rol, input, context

## 9. Testăm două inputuri
Nu vrem să testăm agentul pe un singur exemplu. Un agent RAG trebuie verificat pe mai multe inputuri, ca să vedem dacă păstrează vocea și dacă folosește contextul recuperat.
În acest pas rulăm același agent pe două texte politice scurte.

In [36]:
test_inputs = [
    "CCR a decis anularea alegerilor după suspiciuni privind influențe externe.",
    "Guvernul a anunțat noi măsuri economice care au provocat proteste."
]

In [37]:
def retrieve_context(input_text, k=5):
    query_embedding = model.encode(
        [input_text],
        normalize_embeddings=True
    ).astype("float32")

    scores, positions = index.search(query_embedding, k)

    results = []
    for score, pos in zip(scores[0], positions[0]):
        item = metadata[pos].copy()
        item["score"] = round(float(score), 3)
        results.append(item)

    context_parts = []
    for i, item in enumerate(results, start=1):
        context_parts.append(
            f"""[Fragment {i} | score={item.get("score", "")} | source={item.get("source_channel", "")}]
{item.get("text", "")}
"""
        )

    return results, "\n".join(context_parts)

In [32]:
def generate_response(input_text):
    results, retrieved_context = retrieve_context(input_text, k=K)

    final_prompt = template.format(
        agent_system=role["system"],
        input_text=input_text,
        retrieved_context=retrieved_context
    )

    response = client.chat.completions.create(
        model=MODEL_NAME_LLM,
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0.3
    )

    return {
        "agent_slug": MY_AGENT,
        "agent_name": role["name"],
        "input_text": input_text,
        "retrieved_context": results,
        "prompt": final_prompt,
        "response": response.choices[0].message.content,
        "model": MODEL_NAME_LLM,
        "temperature": 0.3
    }

In [33]:
test_results = []

for text in test_inputs:
    result = generate_response(text)
    test_results.append(result)

    print("=" * 80)
    print("INPUT:")
    print(result["input_text"])
    print("\nRĂSPUNS:")
    print(result["response"])

INPUT:
CCR a decis anularea alegerilor după suspiciuni privind influențe externe.

RĂSPUNS:
Păi normal că se anulează alegerile, că altfel cum să mai fure ei liniștiți și să-și pună pilele la loc? Asta e țara noastră, o cloacă de corupți care ne iau banii pe față și pe dos, iar noi stăm și ne uităm ca proștii!
INPUT:
Guvernul a anunțat noi măsuri economice care au provocat proteste.

RĂSPUNS:
Și ce să protestăm, domnule, că ne fură pe față și ne iau și ultima suflare? Asta e țara lor, nu a noastră, unde ei trăiesc ca în rai pe spatele nostru, al fraierilor care muncesc și plătesc taxe.


# 9. Mini-agent RAG cu tool de regăsire
 
Până acum:
noi am făcut retrieval manual → am pus contextul în prompt → am apelat LLM-ul.
 
Acum:
definim retrieval-ul ca tool → agentul poate folosi tool-ul → apoi generează răspunsul.

In [44]:
!pip install langchain-openai


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
!pip install langchain

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 3.7 MB/s  0:00:00

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 1.3.2

    Uninstalling langchain-core-1.3.2:

      Successfully uninstalled langchain-core-1.3.2

   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-core]
   ----------


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

In [48]:
PROVIDER = "gemini"  # "gemini" sau "deepseek"
if PROVIDER == "gemini":
    MODEL_NAME_AGENT = "gemini-2.5-flash-lite"
    API_KEY = os.getenv("GEMINI_API_KEY")
    BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
elif PROVIDER == "deepseek":
    MODEL_NAME_AGENT = "deepseek-chat"
    API_KEY = os.getenv("DEEPSEEK_API_KEY")
    BASE_URL = "https://api.deepseek.com/v1"
else:
    raise ValueError("Provider necunoscut. Alege 'gemini' sau 'deepseek'.")
 
llm = ChatOpenAI(
    model=MODEL_NAME_AGENT,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.3,
)
print("Provider:", PROVIDER)
print("Model:", MODEL_NAME_AGENT)

Provider: gemini
Model: gemini-2.5-flash-lite


In [49]:
@tool
def retrieve_similar_comments(query: str) -> str:
    """Caută comentarii similare în bula discursivă a agentului."""
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
   
    scores, positions = index.search(query_embedding, K)
    context_parts = []
    for i, (score, pos) in enumerate(zip(scores[0], positions[0]), start=1):
        item = metadata[pos]
        context_parts.append(
            f"""
[Fragment {i} | score={round(float(score), 3)}]
{item.get("text", "")}
"""
        )
    return "\n".join(context_parts)

In [50]:
agent = create_agent(
    model=llm,
    tools=[retrieve_similar_comments],
    system_prompt=role["system"] + """
 
    REGULĂ OBLIGATORIE:
    Înainte să răspunzi, trebuie să folosești instrumentul `retrieve_similar_comments`
    pentru a căuta comentarii similare în corpusul agentului.
 
    Nu răspunde direct fără să folosești instrumentul.
 
După ce primești comentariile similare:
- folosește-le doar ca inspirație de ton și stil;
- nu le copia;
- răspunde cu un singur comentariu;
- maximum 3 propoziții.
"""
)

# Rulăm agentul:

In [51]:
input_text = "CCR a decis anularea alegerilor după suspiciuni privind influențe externe."
agent_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": input_text
        }
    ]
})
print(agent_result["messages"][-1].content)

Decizia CCR de a anula alegerile pe baza unor suspiciuni de influență externă pare o manevră periculoasă, care subminează încrederea în procesul democratic și deschide ușa speculațiilor și manipulării. Este alarmant cum astfel de decizii pot fi folosite pentru a invalida voința populară, sub pretextul unor amenințări invizibile.


In [52]:
# ne uitam daca a folosit tool
for message in agent_result["messages"]:
    print(type(message).__name__)
    print(message)
    print("-" * 80)

HumanMessage
content='CCR a decis anularea alegerilor după suspiciuni privind influențe externe.' additional_kwargs={} response_metadata={} id='fb27633e-4353-4e87-a800-047f99180de8'
--------------------------------------------------------------------------------
AIMessage
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 599, 'total_tokens': 632, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemini-2.5-flash-lite', 'system_fingerprint': None, 'id': 'Yv0Far-TK7iuvdIPxtrhgAs', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e2765-cc06-7ae1-802a-fe315443f199-0' tool_calls=[{'name': 'retrieve_similar_comments', 'args': {'query': 'CCR a decis anularea alegerilor după suspiciuni privind influențe externe'}, 'id': 'function-call-8444966382049850637', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 599, 'output_tok

In [53]:
used_tool = any(
    hasattr(message, "tool_calls") and len(message.tool_calls) > 0
    for message in agent_result["messages"]
)
print("Agentul a folosit tool-ul:", used_tool)

Agentul a folosit tool-ul: True


## 10. Mini-agent RSS: de la știre recentă la comentariu de bulă
Până acum am dat noi manual un text politic agentului.
Acum facem un pas mai agentic: agentul primește acces la două instrumente:
1. un instrument care citește o știre recentă dintr-un feed RSS;
2. un instrument care caută comentarii similare în bula discursivă a agentului.
Fluxul devine:
```text
RSS news → retrieve similar comments → role_XX.yaml → LLM → comentariu de bulă

In [54]:
%pip install -U feedparser

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6104 sha256=645f8706f38476fca361c605664259a5d8d61206ff3d70b464353861150620db
  Stored in directory: c:\users\doina\appdata\local\pip\cache\wheels\3b\25\2a\105d6a15df6914f4d15047691c6c28f9052cc1173e40285d03
Successfully built sgmllib3k

   -------------------- ------------------- 1/2 [feedparser]
   -------------------- ------------------- 1/2 [feedparser]
   -------------------- ------------------- 1/2 [feedparser]
   -------------------- ------------------- 1/2 [feedparser]
   ---------------------------------------- 2/2 [feedparser]

Note: you may need to restart the kernel to use up


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
import feedparser
from langchain_core.tools import tool

### 10.2 Alegem o sursă RSS
Pentru laborator folosim o sursă RSS publică. Poți schimba feed-ul dacă vrei să testezi altă sursă.
Exemple posibile:
 
https://www.g4media.ro/feed
 
https://www.hotnews.ro/rss

In [58]:
#TO DO : alege ce feed vrei
 
RSS_FEED = "https://www.g4media.ro/feed"

### 10.3 Tool 1: citim o știre recentă din RSS
Acest tool ia prima știre din feed și returnează titlul, linkul și rezumatul.
Pentru agent, acest tool este o sursă externă de input.

In [59]:
@tool
def get_latest_news_from_rss() -> str:
    """Ia cea mai recentă știre din feed-ul RSS și returnează titlul, linkul și rezumatul."""
    feed = feedparser.parse(RSS_FEED)
   
    if not feed.entries:
        return "Nu am găsit știri în feed-ul RSS."
   
    entry = feed.entries[0]
   
    title = entry.get("title", "")
    link = entry.get("link", "")
    summary = entry.get("summary", "")
   
    return f"""
TITLU:
{title}
 
LINK:
{link}
 
REZUMAT:
{summary}
"""
 
 
import feedparser
 
RSS_FEED = "https://www.g4media.ro/feed"
 
feed = feedparser.parse(RSS_FEED)
 
print("Număr știri:", len(feed.entries))
feed.entries[0]
 

Număr știri: 10


{'title': 'Se cere închisoare pe viață pentru Vasile Frumuzache, agentul de pază român care a ucis în Italia două escorte / „A acționat cu premeditare”',
 'title_detail': {'type': 'text/plain',
  'language': None,
  'base': 'https://www.g4media.ro/feed',
  'value': 'Se cere închisoare pe viață pentru Vasile Frumuzache, agentul de pază român care a ucis în Italia două escorte / „A acționat cu premeditare”'},
 'links': [{'rel': 'alternate',
   'type': 'text/html',
   'href': 'https://www.g4media.ro/se-cere-inchisoare-pe-viata-pentru-vasile-frumuzache-agentul-de-paza-roman-care-a-ucis-in-italia-doua-escorte-a-actionat-cu-premeditare.html'},
  {'length': '500',
   'type': 'image/jpeg',
   'href': 'https://www.g4media.ro//wp-content/uploads/2026/05/furmuzache.jpeg',
   'rel': 'enclosure'}],
 'link': 'https://www.g4media.ro/se-cere-inchisoare-pe-viata-pentru-vasile-frumuzache-agentul-de-paza-roman-care-a-ucis-in-italia-doua-escorte-a-actionat-cu-premeditare.html',
 'comments': 'https://www.g

In [60]:
# Testăm tool-ul RSS înainte să îl dăm agentului
latest_news = get_latest_news_from_rss.invoke({})
print(latest_news)


TITLU:
Președintele Nicușor Dan transmite că securitatea Europei depinde de unitatea relației transatlantice în ziua în care SUA au suspendat rotația trupelor către Europa

LINK:
https://www.g4media.ro/presedintele-nicusor-dan-transmite-ca-securitatea-europei-depinde-de-unitatea-relatiei-transatlantice-in-ziua-in-care-sua-au-suspendat-rotatia-trupelor-catre-europa.html

REZUMAT:
<p>Președintele Nicușor Dan a transmis joi într-o postare pe rețeaua X că securitatea Europei depinde de unitatea relației cu SUA. El a făcut declarația în ziua în care SUA au suspendat rotația trupelor către Europa, mișcare anunțată de ministrul lituanian al Apărării, și la câteva zile după ce președintele Donald Trump a amenințat că va [&#8230;]</p>
<p>&copy; <a href="https://www.g4media.ro">G4Media.ro</a>.</p>



### TODO — explică ce face tool-ul RSS
Completează:
- `feedparser.parse(RSS_FEED)` face: __________
- `feed.entries[0]` selectează: __________
- Tool-ul returnează trei informații: __________, __________, __________
- De ce este util să testăm tool-ul înainte să îl dăm agentului? __________

In [ ]:
feed = feedparser.parse(RSS_FEED)
 
print("Feed title:", feed.feed.get("title", ""))
print("Număr știri găsite:", len(feed.entries))
 
entry = feed.entries[0]
print("Titlu:", entry.get("title", ""))
print("Link:", entry.get("link", ""))

Feed title: G4Media.ro
Număr știri găsite: 10
Titlu: Se cere închisoare pe viață pentru Vasile Frumuzache, agentul de pază român care a ucis în Italia două escorte / „A acționat cu premeditare”
Link: https://www.g4media.ro/se-cere-inchisoare-pe-viata-pentru-vasile-frumuzache-agentul-de-paza-roman-care-a-ucis-in-italia-doua-escorte-a-actionat-cu-premeditare.html


### 10.4 Tool 2: căutăm comentarii similare în bula agentului
Acest tool reutilizează mecanismul FAISS construit în C5.
Diferența este că acum îl ambalăm ca tool pentru agent.

In [62]:
@tool
def retrieve_similar_comments(query: str) -> str:
    """Caută comentarii similare în bula discursivă a agentului."""
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
   
    scores, positions = index.search(query_embedding, K)
   
    context_parts = []
   
    for i, (score, pos) in enumerate(zip(scores[0], positions[0]), start=1):
        item = metadata[pos]
        fragment = f"""
[Comentariu similar {i} | score={round(float(score), 3)}]
{item.get("text", "")}
"""
        context_parts.append(fragment)
   
    return "\n".join(context_parts)
 
# Testăm tool-ul FAISS separat
test_query = "CCR a decis anularea alegerilor după suspiciuni privind influențe externe."
similar_comments = retrieve_similar_comments.invoke({"query": test_query})
print(similar_comments)
 


[Comentariu similar 1 | score=0.551]
Și atunci de ce nu face contestație dl Georgescu? Ce are a face CCR cu rezultatele alegerilor? Nu are Comisia Electorală competența de a valida alegerile? Dreptul la vot este scris în Constituție.


[Comentariu similar 2 | score=0.372]
Adică UDMR a fost cu Psd tot timpul la guvernare și acum ne mirăm că votează ce mai propune AUR?? Suntem oare așa naivi?


[Comentariu similar 3 | score=0.353]
Calin Georgescu a participat la turul unu ca sa fie anulat si sa nu castige alegerile, asa face el , cere bani ca sa nu i se dea!😅😅😅


[Comentariu similar 4 | score=0.301]
o analiză detaliată, dar trebuie să fim foarte atenți la cum abordăm subiectele politice, mai ales când e vorba de partide și conflicte. Este important să discutăm într-un mod respectuos și informat, având în vedere că astfel de subiecte pot fi foarte sensibile și pot avea un impact puternic asupra opiniei publice. Cum poate opoziția din partidu AUR să încerce să erodeze democrația? Manipula

### TODO — explică tool-ul de regăsire
Completează:
- Acest tool primește ca input: un text / o întrebare nouă
- Transformă inputul în: embedding vectorial
- Caută în: vectorstore / baza de comentarii similare
- Returnează: comentarii similare cu scor de similaritate
- De ce acest tool este diferit de simpla generare cu LLM? pentru că recuperează context real din date înainte să genereze răspunsul

### 10.5 Creăm agentul cu două instrumente
Agentul are acum:
- rolul discursiv din `role_XX.yaml`;
- un tool pentru știri recente;
- un tool pentru comentarii similare.
Instrucțiunea importantă: agentul trebuie să folosească mai întâi RSS-ul, apoi regăsirea semantică.

In [63]:
agent_news = create_agent(
    model=llm,
    tools=[get_latest_news_from_rss, retrieve_similar_comments],
    system_prompt=role["system"] + """
 
Ai două instrumente:
1. get_latest_news_from_rss — citește o știre recentă dintr-un feed RSS.
2. retrieve_similar_comments — caută comentarii similare în bula discursivă.
 
REGULĂ OBLIGATORIE:
Folosește mai întâi get_latest_news_from_rss.
Apoi folosește retrieve_similar_comments pe titlul sau rezumatul știrii.
 
După ce ai primit ambele rezultate, scrie:
 
ȘTIRE FOLOSITĂ:
titlul știrii și linkul
 
COMENTARIU:
un singur comentariu de YouTube, maximum 3 propoziții, în vocea agentului
 
NOTĂ:
o propoziție scurtă despre ce a venit din știre și ce a venit din bula discursivă.
 
Nu prezenta interpretarea agentului ca fapt verificat.
"""
)

### 10.6 Rulăm mini-agentul RSS
Acum nu mai scriem noi inputul politic.
Îi cerem agentului să ia o știre recentă și să o comenteze.

In [64]:
agent_news_result = agent_news.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Alege o știre recentă din RSS și comenteaz-o în vocea agentului."
        }
    ]
})
 
print(agent_news_result["messages"][-1].content)

ȘTIRE FOLOSITĂ:
Președintele Nicușor Dan transmite că securitatea Europei depinde de unitatea relației transatlantice în ziua în care SUA au suspendat rotația trupelor către Europa
https://www.g4media.ro/presedintele-nicusor-dan-transmite-ca-securitatea-europei-depinde-de-unitatea-relatiei-transatlantice-in-ziua-in-care-sua-au-suspendat-rotatia-trupelor-catre-europa.html

COMENTARIU:
Declarația lui Nicușor Dan despre importanța relației transatlantice vine ca o reacție firească la semnalele de alarmă din SUA, arătând că unitatea europeană nu e un moft, ci o necesitate. E clar că unii preferă să flirteze cu izolaționismul, dar istoria ne-a învățat deja unde duce asta.

NOTĂ:
Știrea subliniază o poziție pro-europeană și pro-occidentală, iar comentariile similare reflectă o tendință de respingere a acestor mesaje și o preferință pentru discursuri naționaliste sau izolaționiste.


### 10.7 Verificăm dacă agentul a folosit instrumentele
Un agent cu tool-uri trebuie verificat.
Nu este suficient să vedem răspunsul final. Trebuie să vedem dacă a apelat instrumentele.

In [65]:
for message in agent_news_result["messages"]:
    print(type(message).__name__)
   
    if hasattr(message, "tool_calls"):
        print("tool_calls:", message.tool_calls)
   
    print(str(message.content)[:1200])
    print("-" * 80)

HumanMessage
Alege o știre recentă din RSS și comenteaz-o în vocea agentului.
--------------------------------------------------------------------------------
AIMessage
tool_calls: [{'name': 'get_latest_news_from_rss', 'args': {}, 'id': 'function-call-8011815146373109698', 'type': 'tool_call'}]

--------------------------------------------------------------------------------
ToolMessage

TITLU:
Președintele Nicușor Dan transmite că securitatea Europei depinde de unitatea relației transatlantice în ziua în care SUA au suspendat rotația trupelor către Europa

LINK:
https://www.g4media.ro/presedintele-nicusor-dan-transmite-ca-securitatea-europei-depinde-de-unitatea-relatiei-transatlantice-in-ziua-in-care-sua-au-suspendat-rotatia-trupelor-catre-europa.html

REZUMAT:
<p>Președintele Nicușor Dan a transmis joi într-o postare pe rețeaua X că securitatea Europei depinde de unitatea relației cu SUA. El a făcut declarația în ziua în care SUA au suspendat rotația trupelor către Europa, mișcare an

In [73]:
used_tools = []
 
for message in agent_news_result["messages"]:
    if hasattr(message, "tool_calls"):
        for call in message.tool_calls:
            used_tools.append(call["name"])
 
print("Tool-uri folosite:", used_tools)
print("A folosit RSS:", "get_latest_news_from_rss" in used_tools)
print("A folosit FAISS:", "retrieve_similar_comments" in used_tools)


Tool-uri folosite: ['get_latest_news_from_rss', 'retrieve_similar_comments']
A folosit RSS: True
A folosit FAISS: True


### TODO — concluzie scurtă
Scrie 3–4 fraze:
1. Ce a făcut agentul diferit față de varianta manuală? (aici putem răspunde doar in gând)
1. Ce ar trebui verificat de un om înainte ca acest răspuns să fie folosit într-o aplicație publică?


Agentul a folosit comentarii similare din corpus și informații din RSS pentru a genera un răspuns mai apropiat de stilul comunității simulate. Față de varianta manuală, răspunsul a fost mai contextual și mai coerent cu tonul agentului definit în roles.yaml. Înainte ca răspunsul să fie folosit într-o aplicație publică, un om ar trebui să verifice corectitudinea informațiilor, tonul folosit și eventualele afirmații manipulative sau părtinitoare.

In [71]:
from pathlib import Path
import yaml

path = Path("assets/roles/roles.yaml")

try:
    with open(path, "r", encoding="utf-8") as f:
        roles = yaml.safe_load(f)

    print(roles.keys())
    print(roles["agents"].keys())

except yaml.YAMLError as e:
    print(e)

while parsing a block mapping
  in "assets\roles\roles.yaml", line 20, column 1
expected <block end>, but found '<block mapping start>'
  in "assets\roles\roles.yaml", line 60, column 3


In [72]:
from pathlib import Path
import yaml

path = Path("assets/roles/roles.yaml")

try:
    with open(path, "r", encoding="utf-8") as f:
        roles = yaml.safe_load(f)

    print(roles.keys())
    print(roles["agents"].keys())

except yaml.YAMLError as e:
    print(e)

dict_keys(['agents'])
dict_keys(['anti_suveranist', 'conspirationist'])
